# KCD 2026 · 외부 검증 (다른 병원)

이 노트북은 **모델 패키지 폴더 안에서** 실행합니다(`kcd_liver/`, `models/`가 옆에 있어야 함). 학습은 하지 않고, MIMIC-IV에서 학습해 고정한 4개 모델을 이 병원 자료에 그대로 적용해 성능을 계산합니다.

| 모델 | 실행 시점 | 출력 |
|---|---|---|
| 1 Agent 3 | 정상일 때 매시간 | 24시간 이내 간기능 이상 위험 |
| 2 Agent 2 | 첫 이상 시점 | 예후 4경로 (해소 × ICU 사망, 추적은 ICU 퇴실까지) |
| 3 Agent 1 | 첫 이상 시점 | 원인 10개 0/1 (규칙 코드) + 보조 군집 이식 |
| 4 궤적 | 상태와 무관, 매시간 | ALT·AST·ALP·TBIL 24시간 예측 (10·50·90%) |

**자료 형식**: MIMIC-IV와 같은 폴더·열 구조(`hosp/patients, admissions, labevents, d_labitems, diagnoses_icd, emar`, `icu/icustays, chartevents, inputevents, d_items, outputevents`). 검사·활력징후 itemid가 다르면 `notebooks/KCD_MIMICIV_liver_agents_v2.ipynb`의 `LABS`·`VITAL_ITEMS` 대응표를 이 병원 코드로 바꿔야 합니다.

**돌려주는 것**: `kcd_external_return_*.zip` (집계만, 10 미만 칸 가림). 환자 단위 파일은 `external_work/`에만 남습니다.

**버전**: 모델 파일은 학습 때와 같은 scikit-learn 버전에서 읽어야 합니다. `MODEL_CARD.md`의 버전을 확인하세요.

In [ ]:
# ===== 설정 =====
from pathlib import Path
import os, sys, json, time, shutil, zipfile, traceback, subprocess, platform
from datetime import datetime
REPO = Path.cwd()
assert (REPO / 'kcd_liver').is_dir() and (REPO / 'models').is_dir(), '모델 패키지 폴더 안에서 실행하세요'
DATA_ROOT = Path(r'D:/hospital_data/mimic_format')   # 이 병원 자료 (hosp/ icu/)
SITE_NAME = 'external_site'
MAX_PATIENTS = None
MODEL_DIR = REPO / 'models'
WORK = REPO / 'external_work'
sys.path.insert(0, str(REPO)); WORK.mkdir(exist_ok=True)
REPORT = {'started_at': datetime.now().isoformat(timespec='seconds'), 'steps': {}, 'python': platform.python_version(), 'site': SITE_NAME}
print('DATA_ROOT 접근 가능:', DATA_ROOT.exists())
card = REPO / 'MODEL_CARD.md'
if card.exists(): print([l for l in card.read_text(encoding='utf-8').splitlines() if 'scikit-learn' in l][:1])

In [ ]:
# ===== 환경 확인 =====
need = ['pandas', 'numpy', 'sklearn', 'matplotlib', 'joblib', 'nbformat', 'nbclient', 'ipykernel']
missing = []
for m in need:
    try: __import__(m)
    except ImportError: missing.append(m)
if missing:
    print('설치가 필요합니다:', missing)
    print('pip install', ' '.join('scikit-learn' if m == 'sklearn' else m for m in missing))
    raise SystemExit('위 패키지를 설치하고 커널을 재시작한 뒤 다시 실행하세요.')
import pandas as pd, numpy as np, sklearn
REPORT['versions'] = {'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__}
print(REPORT['versions'])

In [ ]:
# ===== 도우미: 분석 노트북을 설정만 바꿔 실행 =====
import nbformat
from nbclient import NotebookClient
MAIN_NB = REPO / 'notebooks' / 'KCD_MIMICIV_liver_agents_v2.ipynb'
ORIG_ROOT = "ROOT = Path(r'\\\\10.7.12.146\\데이터 및 소스\\MIMIC\\MIMIC4-V3.1')"

def run_main(name, root, source, max_patients=None, extra=None, env=None):
    work = WORK / name; work.mkdir(parents=True, exist_ok=True)
    nb = nbformat.read(MAIN_NB, as_version=4)
    subs = [(ORIG_ROOT, f"ROOT = Path(r'{root}')"), ("DATA_SOURCE = 'mimic-iv'", f"DATA_SOURCE = '{source}'"),
            ("MAX_PATIENTS = None  # all patients", f"MAX_PATIENTS = {max_patients!r}  # all patients")] + list((extra or {}).items())
    hit = {a: 0 for a, _ in subs}
    for c in nb.cells:
        for a, b in subs:
            if a in c.source: c.source = c.source.replace(a, b); hit[a] += 1
    miss = [a for a, n in hit.items() if n == 0]
    if miss: raise RuntimeError(f'설정 줄을 찾지 못했습니다: {miss}')
    env = {'KCD_REPO': REPO, **(env or {})}
    old = {k: os.environ.get(k) for k in (env or {})}
    os.environ.update({k: str(v) for k, v in (env or {}).items()})
    t = time.time()
    try:
        code_idx = [i for i, cc in enumerate(nb.cells) if cc.cell_type == 'code']
        t_last = [time.time()]
        timing = REPORT.setdefault('cell_minutes', {}).setdefault(name, [])
        def _done(cell, cell_index, execute_reply=None, **kw):  # live progress + per-cell time for the run report
            dt = (time.time() - t_last[0]) / 60; t_last[0] = time.time()
            head = next((l.strip() for l in cell.source.splitlines() if l.strip()), '')[:60]
            k = code_idx.index(cell_index) + 1 if cell_index in code_idx else cell_index
            timing.append({'cell': cell_index, 'minutes': round(dt, 2), 'head': head})
            print(f'[{name}] {k}/{len(code_idx)} 완료 ({dt:.1f}분, 누적 {(time.time() - t) / 60:.1f}분): {head}', flush=True)
        try:
            client = NotebookClient(nb, timeout=None, kernel_name='python3', resources={'metadata': {'path': str(work)}},
                                    on_cell_executed=_done)
        except Exception:  # very old nbclient without hooks: run without progress lines
            client = NotebookClient(nb, timeout=None, kernel_name='python3', resources={'metadata': {'path': str(work)}})
        import threading
        plog = work / 'progress.log'
        if plog.exists(): plog.unlink()
        stop = threading.Event()
        def _tail():  # stream the child notebook's progress lines (file reads, chunk counts) while it runs
            pos = 0
            while not stop.wait(15):
                try:
                    with open(plog, encoding='utf-8') as f:
                        f.seek(pos); new = f.read(); pos = f.tell()
                    for l in new.splitlines(): print(f'[{name}] {l}', flush=True)
                except OSError: pass
        th = threading.Thread(target=_tail, daemon=True); th.start()
        try: client.execute()
        finally: stop.set(); th.join(timeout=5)
    finally:
        nbformat.write(nb, work / 'executed_PRIVATE.ipynb')   # 환자 정보가 출력될 수 있어 압축에서 제외
        for k, v in old.items():
            if v is None: os.environ.pop(k, None)
            else: os.environ[k] = v
    return work / 'kcd_liver_v2_outputs', round((time.time() - t) / 60, 1)

def step(name, fn):
    t = time.time()
    try:
        out = fn(); REPORT['steps'][name] = {'status': 'ok', 'minutes': round((time.time() - t) / 60, 1), **(out or {})}
    except BaseException as ex:
        REPORT['steps'][name] = {'status': 'failed', 'minutes': round((time.time() - t) / 60, 1), 'error': f'{type(ex).__name__}: {ex}',
                                 'trace_tail': traceback.format_exc()[-1500:]}
        print('실패:', name, ex)
    print(name, REPORT['steps'][name]['status'], REPORT['steps'][name]['minutes'], '분')

In [ ]:
# ===== 1단계: 이 병원 자료로 분석 노트북 실행 (코호트·사건·원인 규칙; 사이트 내보내기 끔) =====
EXT_OUT = WORK / 'external' / 'kcd_liver_v2_outputs'
def e1():
    out, m = run_main('external', DATA_ROOT, 'external', MAX_PATIENTS, extra={'EXPORT_SITE=True': 'EXPORT_SITE=False'},
                      env={'KCD_MODEL_DIR': MODEL_DIR})
    return {'notebook_minutes': m, 'cohort_flow': pd.read_csv(out / '04_cohort_flow.csv').to_dict('records')}
step('1_external_notebook', e1)

In [ ]:
# ===== 2단계: 고정 모델 적용, 성능 계산 =====
def e2():
    from scripts.external_validate import main as ev
    ev(['--out-dir', str(EXT_OUT), '--model-dir', str(MODEL_DIR), '--data-root', str(DATA_ROOT), '--site', SITE_NAME])
    R = json.loads((EXT_OUT / 'results_external.json').read_text(encoding='utf-8'))
    return {'agent3_auroc': (R.get('agent3') or {}).get('auroc'), 'prognosis_n': (R.get('prognosis') or {}).get('n')}
step('2_external_validation', e2)

In [ ]:
# ===== 3단계: 돌려줄 파일 (집계만) =====
FORBIDDEN_KEYS = {'subject_id', 'hadm_id', 'stay_id', 'labevent_id', 'case_id'}
def scan_ids(o, hits=None):
    hits = [] if hits is None else hits
    if isinstance(o, dict):
        for k, v in o.items():
            if k in FORBIDDEN_KEYS: hits.append(k)
            scan_ids(v, hits)
    elif isinstance(o, list):
        for v in o: scan_ids(v, hits)
    return hits
def e3():
    R = json.loads((EXT_OUT / 'results_external.json').read_text(encoding='utf-8'))
    if scan_ids(R): raise RuntimeError('환자 식별 키가 있어 압축하지 않습니다')
    zp = Path.cwd() / f"kcd_external_return_{datetime.now().strftime('%Y%m%d_%H%M')}.zip"
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(EXT_OUT / 'results_external.json', 'results_external.json')
        for n in ['04_cohort_flow.csv', '02_lab_quality.csv']:
            if (EXT_OUT / n).exists(): z.write(EXT_OUT / n, 'aggregates/' + n)
        z.writestr('run_report.json', json.dumps(REPORT, ensure_ascii=False, indent=1, default=str))
    print('완성:', zp); return {'package': str(zp)}
step('3_package', e3)
print(json.dumps({k: v.get('status') for k, v in REPORT['steps'].items()}, ensure_ascii=False))